# 📓 Notebook 1 — Data Ingestion & Vector Store
### Egyptian Telecom RAG Support Assistant

This notebook:
1. Mounts Google Drive
2. Loads multi-operator telecom documentation (PDF/TXT) for WE, Vodafone, and Etisalat
3. Chunks documents with `RecursiveCharacterTextSplitter` (chunk_size=1000, chunk_overlap=150)
4. Generates embeddings using `OllamaEmbeddings` (model: `nomic-embed-text`)
5. Persists the vector store into ChromaDB on Google Drive

**Run this notebook once** (re-run only when your source documents change).

## Cell 1 — Install dependencies

In [ ]:
# Update apt's package index first, then install zstd (required by the Ollama installer).
# Skipping `apt-get update` is the usual reason this install silently fails on a fresh Colab VM.
!apt-get update -qq
!apt-get install -y zstd -qq

import shutil
if shutil.which("zstd") is None:
    raise RuntimeError(
        "zstd did not install correctly via apt-get. Try re-running this cell — "
        "if it still fails, run `!apt-get update && !apt-get install -y zstd` manually "
        "in a new cell and check the output for errors."
    )
print("✅ zstd installed at:", shutil.which("zstd"))

# Python dependencies (unstructured removed — not needed for PyPDF/Text loaders,
# and its pin doesn't support newer Colab Python versions)
!pip install -q langchain==0.3.7 langchain-community==0.3.7 langchain-chroma==0.1.4 \
    chromadb==0.5.20 pypdf==5.1.0 sentence-transformers==3.3.1

# Install Ollama (now that zstd is confirmed present)
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time

# Sanity check: confirm the ollama binary actually installed before trying to run it
ollama_path = shutil.which("ollama")
if ollama_path is None:
    raise RuntimeError(
        "Ollama installation failed — 'ollama' binary not found on PATH. "
        "Scroll up in this cell's output for the actual install error."
    )
print(f"✅ Ollama installed at: {ollama_path}")

subprocess.Popen([ollama_path, "serve"])
time.sleep(5)

# NOTE: llama3.2 is a chat/generation model, NOT an embeddings model — Ollama's backend
# rejects embedding requests for it. We use nomic-embed-text here instead, which is
# purpose-built for embeddings. Notebook 2 (the backend) must use this SAME embedding
# model when loading this vector store, or similarity search will be meaningless.
!{ollama_path} pull nomic-embed-text

## Cell 2 — Mount Google Drive & prepare folder structure

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Root folder structure expected on your Drive:
# /content/drive/MyDrive/TelecomRAG/docs/WE/*.pdf
# /content/drive/MyDrive/TelecomRAG/docs/Vodafone/*.pdf
# /content/drive/MyDrive/TelecomRAG/docs/Etisalat/*.pdf
BASE_DIR = "/content/drive/MyDrive/TelecomRAG"
DOCS_DIR = os.path.join(BASE_DIR, "docs")
VECTOR_DIR = os.path.join(BASE_DIR, "chroma_db")

for company in ["WE", "Vodafone", "Etisalat"]:
    os.makedirs(os.path.join(DOCS_DIR, company), exist_ok=True)
os.makedirs(VECTOR_DIR, exist_ok=True)

print(f"Docs directory: {DOCS_DIR}")
print(f"Vector store directory: {VECTOR_DIR}")
print("⚠️ Place each operator's PDFs/TXT files inside their respective subfolder, then continue.")

## Cell 2.5 — Upload documents directly from your computer (optional)

Run this cell once per operator whose files you want to upload. It opens a file picker,
lets you choose one or more PDFs/TXT files from your local machine, and copies them into
the correct `TelecomRAG/docs/<company>/` folder on Drive.

Skip this cell entirely if you'd rather drag files into the Drive folders manually via
the Drive website — either approach works, and Cell 3 below reads from the same folders.

In [ ]:
from google.colab import files
import shutil

def upload_documents_for_operator(company_name: str):
    """Opens a local file picker and copies the selected files into the operator's Drive folder."""
    if company_name not in ["WE", "Vodafone", "Etisalat"]:
        raise ValueError("company_name must be one of: WE, Vodafone, Etisalat")

    target_folder = os.path.join(DOCS_DIR, company_name)
    os.makedirs(target_folder, exist_ok=True)

    print(f"📤 Select PDF/TXT file(s) for {company_name}. A file picker will open below...")
    uploaded = files.upload()

    if not uploaded:
        print("⚠️ No files were selected.")
        return

    saved_paths = []
    for filename, content in uploaded.items():
        ext = os.path.splitext(filename)[1].lower()
        if ext not in [".pdf", ".txt"]:
            print(f"  ⚠️ Skipping '{filename}' — only .pdf and .txt files are supported.")
            continue

        dest_path = os.path.join(target_folder, filename)
        with open(dest_path, "wb") as f:
            f.write(content)
        saved_paths.append(dest_path)
        print(f"  ✔ Saved to {dest_path}")

    print(f"\n✅ {len(saved_paths)} file(s) uploaded for {company_name}.")
    return saved_paths


# 👇 Run this line (or copy it into a new cell) for EACH operator you have documents for.
# Each call opens its own file picker — select all files for that operator, then re-run
# this cell (or a copy of it) for the next operator.

upload_documents_for_operator("WE")
# upload_documents_for_operator("Vodafone")
# upload_documents_for_operator("Etisalat")

## Cell 3 — Load & tag documents per operator

In [ ]:
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.schema import Document
import glob

def load_operator_documents(company_name: str, folder_path: str) -> list[Document]:
    documents = []
    pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))
    txt_files = glob.glob(os.path.join(folder_path, "*.txt"))

    for pdf_path in pdf_files:
        try:
            loader = PyPDFLoader(pdf_path)
            pages = loader.load()
            for page in pages:
                page.metadata["company"] = company_name
                page.metadata["source"] = os.path.basename(pdf_path)
                page.metadata["file_type"] = "pdf"
            documents.extend(pages)
            print(f"  ✔ Loaded {len(pages)} pages from {os.path.basename(pdf_path)}")
        except Exception as e:
            print(f"  ✘ Failed to load {pdf_path}: {e}")

    for txt_path in txt_files:
        try:
            loader = TextLoader(txt_path, encoding="utf-8")
            docs = loader.load()
            for doc in docs:
                doc.metadata["company"] = company_name
                doc.metadata["source"] = os.path.basename(txt_path)
                doc.metadata["file_type"] = "txt"
            documents.extend(docs)
            print(f"  ✔ Loaded {os.path.basename(txt_path)}")
        except Exception as e:
            print(f"  ✘ Failed to load {txt_path}: {e}")

    return documents

all_documents = []
for company in ["WE", "Vodafone", "Etisalat"]:
    print(f"\nProcessing {company}...")
    folder = os.path.join(DOCS_DIR, company)
    docs = load_operator_documents(company, folder)
    all_documents.extend(docs)

print(f"\n✅ Total raw documents loaded: {len(all_documents)}")

if len(all_documents) == 0:
    print("⚠️ No documents found. Add PDFs/TXT files to the operator folders in Drive and re-run this cell.")

## Cell 4 — Chunk documents

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunked_documents = text_splitter.split_documents(all_documents)

# Ensure every chunk retains its metadata and gets a stable chunk id
for idx, chunk in enumerate(chunked_documents):
    chunk.metadata["chunk_id"] = idx

print(f"✅ Total chunks created: {len(chunked_documents)}")
if chunked_documents:
    print("\nSample chunk metadata:", chunked_documents[0].metadata)
    print("Sample chunk text (first 200 chars):", chunked_documents[0].page_content[:200])

## Cell 5 — Generate embeddings & persist to ChromaDB on Drive

In [ ]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_chroma import Chroma

# Using nomic-embed-text instead of llama3.2 for embeddings — llama3.2 is a chat model
# and Ollama's backend rejects embedding calls for it ("does not support embeddings").
# nomic-embed-text is purpose-built for this. llama3.2 is still used as the LLM for
# answer generation in Notebook 2 — only the embedding model changes here.
embeddings = OllamaEmbeddings(model="nomic-embed-text")

print("Building vector store... this may take a while depending on document volume.")

vectorstore = Chroma.from_documents(
    documents=chunked_documents,
    embedding=embeddings,
    persist_directory=VECTOR_DIR,
    collection_name="telecom_support_kb"
)

print(f"✅ Vector store persisted at: {VECTOR_DIR}")
print(f"✅ Total vectors stored: {vectorstore._collection.count()}")

## Cell 6 — Sanity check retrieval

In [ ]:
test_query = "How do I reset my router?"
results = vectorstore.similarity_search(test_query, k=3)

for i, r in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Company:", r.metadata.get("company"))
    print("Source:", r.metadata.get("source"))
    print("Content snippet:", r.page_content[:200])